<a href="https://colab.research.google.com/github/jlhelling/s2-dw-wof/blob/main/S2-DW-WOF_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Global water occurrence frequency (WOF) mapping from Sentinel-2**

This workflow enables to create *monthly and annual water occurrence frequency (WOF) maps* from full Sentinel-2 timeseries for any region of interest and export the to Google Drive.

Developed by **Leo Helling**

*UMR 5600 - Environment, Ville, Société (EVS) lab*,
*Centre National de Recherche Scientifique (CNRS)*,
*ENS de Lyon*,
*France*

please refer to the following publication for any data created from this workflow:

Helling, L., Singh, S., Rey, L., Parmentier, H., Messager, M., Piégay, H., Belletti, B. (2026). *Mapping water in a Dynamic World: Annual Water Occurrence Frequencies from full Sentinel-2 timeseries* (in preparation)

Contact: leo.helling@ens-lyon.fr

## Step 1: Install required libraries and load functions

In [4]:
%%capture
!pip install --upgrade ee geemap

In [ ]:
# @param

import ee
import geemap

### Helper functions ### ----------------------------------------------------------------------------
def is_image_aoi(AOI) -> bool:
    """Return True if AOI is an ee.Image mask (as opposed to a vector AOI)."""
    return isinstance(AOI, ee.image.Image)


def normalize_aoi(AOI):
    """Normalize an AOI input into a (mask_image, geometry) tuple.

    The workflow historically expects an AOI *image mask* (so it can call
    `.updateMask(AOI)` and `.geometry()`). For vector AOIs (Geometry / Feature /
    FeatureCollection), we generate an equivalent constant mask image.

    Supported inputs:
    - `ee.Image`: non-zero pixels are treated as inside the AOI
    - `ee.Geometry`, `ee.Feature`, `ee.FeatureCollection`

    Returns:
        (ee.Image, ee.Geometry):
            - mask image with value 1 inside AOI and masked outside
            - AOI geometry
    """

    # Image AOI: treat non-zero pixels as inside
    if is_image_aoi(AOI):
        aoi_img = ee.Image(AOI)
        first_band = ee.String(aoi_img.bandNames().get(0))
        inside = aoi_img.select([first_band]).neq(0)
        # Keep the source projection by masking the derived image itself.
        mask_img = inside.selfMask().rename('AOI')
        return mask_img, aoi_img.geometry()

    # Vector AOI: Geometry / Feature / FeatureCollection
    if isinstance(AOI, ee.geometry.Geometry):
        geom = ee.Geometry(AOI)
    elif isinstance(AOI, ee.feature.Feature):
        geom = ee.Feature(AOI).geometry()
    elif isinstance(AOI, ee.featurecollection.FeatureCollection):
        geom = ee.FeatureCollection(AOI).geometry()
    else:
        # Best-effort cast: this covers cases where callers pass a list of
        # features, a table asset ID, or a ComputedObject that is a collection.
        geom = ee.FeatureCollection(AOI).geometry()

    mask_img = ee.Image.constant(1).clip(geom).selfMask().rename('AOI')
    return mask_img, geom

def filter_by_year_and_month(COL, YEAR, MONTH):
    """
    Filters an Earth Engine ImageCollection by a specific year and month.
    Args:
        COL (ee.ImageCollection): The ImageCollection to filter.
        year (int): The year to filter by.
        month (int): The month to filter by (1-12).
    Returns:
        ee.ImageCollection: The filtered ImageCollection containing images from the specified year and month.
    """
    start_date = ee.Date.fromYMD(YEAR, MONTH, 1)
    end_date = start_date.advance(1, 'month')
    return COL.filterDate(start_date, end_date)


def apply_cloud_filtering(COL, ROI_GEO, DATE_START, DATE_END, CLOUD_THRESHOLD = 0.7):
    """
    Applies cloud filtering to a given Sentinel-2 image collection using the Cloud Score Plus Collection.
    Args:
        COL (ee.ImageCollection): The input Sentinel-2 image collection to be filtered.
        ROI_GEO (ee.Geometry): The region of interest for filtering the collection.
        DATE_START (str): The start date for filtering in the format 'YYYY-MM-DD'.
        DATE_END (str): The end date for filtering in the format 'YYYY-MM-DD'.
        CLOUD_THRESHOLD (float, optional): The cloud score threshold for filtering. Default is 0.7.
    Returns:
        ee.ImageCollection: The cloud-filtered image collection.
    """
    # Function to apply the cloudscore threshold mask
    def apply_cloud_mask(img):
        return img.updateMask(img.select('cs').gte(CLOUD_THRESHOLD))

    # Load Cloud Score Plus Collection for cloud filtering, select only cs band
    csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')\
        .filterBounds(ROI_GEO)\
        .filterDate(DATE_START, DATE_END)\
        .select('cs')

    # # filter cs collection by cloud threshold
    # csPlus_filtered = csPlus.map(apply_cloud_mask)

    # link COLLECTION and CS+ to remove unclear pixels below cs-threshold
    return COL.linkCollection(csPlus, ['cs']).map(apply_cloud_mask)

# Function to calculate the water index and remove GSW-pixels
def calc_water(img):
    """calculate water index by applying NDWI, combining it with water DW water probability and normalizing to values [0,1]."""

    ndwi = img.normalizedDifference(['B3', 'B8']).add(1).divide(2).rename('NDWI')

    # Combine NDWI and water probability
    water_prob = img.select('water')

    # compute polynomial threshold using NDWI
    ndwi_val = ndwi.select('NDWI')
    ndwi_sq = ndwi_val.multiply(ndwi_val)
    threshold = (
        ndwi_val.multiply(-1.82940789473684)
        .add(ndwi_sq.multiply(0.867083333333334))
        .add(1.08532894736842)
    )

    # DW water probability adjusted by NDWI polynomial
    combi = water_prob.add(ndwi_val).rename('water_combi')
    water_mask = water_prob.gte(threshold).rename('water_mask').unmask(0)

    return img.addBands(water_mask).addBands(combi).addBands(ndwi)

def load_fabdem(ROI_GEO, smooth=True):
    """
    Loads the FABDEM (Forest And Buildings removed Copernicus DEM) elevation data for a specified region of interest (ROI) and coordinate reference system (CRS), with an option to apply Gaussian smoothing.
    Args:
        ROI_GEO (ee.Geometry): The region of interest as an Earth Engine Geometry object.
        smooth (bool, optional): Whether to apply Gaussian smoothing to the DEM. Defaults to True.
    Returns:
        ee.Image: The processed FABDEM elevation image, either smoothed or raw, clipped to the ROI and projected to the specified CRS.
    """

    # Load DEM
    fabdem = ee.ImageCollection("projects/sat-io/open-datasets/FABDEM")
    fabdem_elev = fabdem.filterBounds(ROI_GEO).mosaic().clip(ROI_GEO).reproject(crs='EPSG:3857', scale=30)

    # Smooth the elevation model
    smoothedDEM = fabdem_elev.convolve(ee.Kernel.gaussian(
        radius = 60, # Larger radius for more smoothing
        sigma = 30,  # Standard deviation
        units = 'meters'))

    if smooth:
        return smoothedDEM
    else:
        return fabdem_elev

def model_shadow_mask(DEM, AZIMUTH, ZENITH, expand=True):
    """
    Generates a shadow mask from a Digital Elevation Model (DEM) using sun azimuth and zenith angles.
    This function applies the hillShadow mask from GEE to identify shadowed areas in the DEM based on the provided sun position.
    Optionally, the shadow mask can be expanded to cover neighboring pixels.
    Args:
        DEM (ee.Image): The digital elevation model as an Earth Engine Image.
        AZIMUTH (float or ee.Number): Sun azimuth angle in degrees.
        ZENITH (float or ee.Number): Sun zenith angle in degrees.
        expand (bool, optional): If True, expands the shadow mask to include neighboring pixels. Defaults to True.
    Returns:
        ee.Image: An Earth Engine Image with a single band named 'shadow_mask', where shadowed areas are marked as 1 and others as 0.
    """
    # apply hillShadow()-mask to remove pixels in shadow areas
    sunAzimuth = ee.Number(AZIMUTH)
    sunZenith = ee.Number(ZENITH)

    # # Calculate the hill shadow mask and mark only shadow areas with 1 and the rest as 0
    # We force the shadow algorithm to run in EPSG:3857 - This aligns with the GEE documentation requirement for Mercator projections.
    shadow_areas = ee.Terrain.hillShadow(
        DEM.reproject(crs='EPSG:3857', scale=30),
        sunAzimuth,
        sunZenith,
        100, # Neighborhood size (increase if high mountains are nearby)
        True # Hysteresis
    ).eq(0).unmask(0)

    if expand:
        # Post-processing: Expand shadows slightly
        shadow_areas_exp = shadow_areas.focal_max(2, 'circle', 'pixels').unmask(0)
        return shadow_areas_exp.rename('monthly_shadow_mask')
    else:
        return shadow_areas.rename('monthly_shadow_mask')


def load_s2_dw_water(ROI, YEAR_MIN, YEAR_MAX, FILTER_CLOUDS=True, CLOUD_THRESHOLD=0.60):
    """
    Loads and processes Sentinel-2 and Dynamic World water data for a specified region and time range.
    This function retrieves Dynamic World water probability images and links them with corresponding Sentinel-2
    surface reflectance (SR) or top-of-atmosphere (TOA) images. It applies spatial, temporal, and optional cloud
    filtering, and computes water-related bands and masks.
    Args:
        ROI (ee.Geometry): The region of interest as an Earth Engine geometry.
        YEAR_MIN (int or str): The starting year (inclusive) for the time range.
        YEAR_MAX (int or str): The ending year (inclusive) for the time range.
        FILTER_CLOUDS (bool, optional): Whether to apply cloud filtering using CloudScore+. Defaults to True.
        CLOUD_THRESHOLD (float, optional): Cloud probability threshold for filtering. Defaults to 0.60.
    Returns:
        ee.ImageCollection: An Earth Engine ImageCollection containing water probability, NDWI, water mask,
        and combined water mask bands for the specified region and time range, with optional cloud filtering applied.
    Notes:
        - The returned collection contains the bands: 'water', 'label', 'NDWI', 'water_mask', and 'water_combi'.
    """


    # Function to set 'has_data' property based on the count of pixels in the image
    # This function checks if the image has any data within the ROI and sets 'has_data
    def set_has_data(img):
        count = img.reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=ROI,
            scale=10,
            maxPixels=1e13
        ).values().get(0)
        # If count is None, set has_data to False
        has_data = ee.Algorithms.If(ee.Number(count).gt(0), True, False)

        return img.set('has_data', has_data)

    # Function to link Dynamic World images with Sentinel-2 images
    # This function retrieves the corresponding Sentinel-2 image based on the system:index of the Dynamic World image
    # and adds the necessary bands (B3, B8) along with solar angles to the Dynamic World image.
    # If the Sentinel-2 surface reflectance image is not available, it falls back to the TOA image.
    def get_s2_image(dw_img):
        index = dw_img.get('system:index')

        # Get the corresponding Sentinel-2 image from SR or TOA collection
        sr_img = S2_SR.filter(ee.Filter.eq('system:index', index)).first()
        toa_img = S2_TOA.filter(ee.Filter.eq('system:index', index)).first()

        # If SR image is not available, use TOA image
        s2_img = ee.Image(ee.Algorithms.If(sr_img,
                                           sr_img,
                                           toa_img
                                        #    toa_img.select(['B3', 'B8']).addBands(
                                        #         toa_img.select('B3').add(0.07).rename('B3_adj')
                                        #     ).select(['B3_adj', 'B8']).rename(['B3', 'B8'])
                                        # see: https://www.sciencedirect.com/science/article/pii/S0034425718301883?
                                           ))

        linked = dw_img.addBands(s2_img.select(['B3', 'B8'])) \
            .set({
                'MEAN_SOLAR_AZIMUTH_ANGLE': s2_img.get('MEAN_SOLAR_AZIMUTH_ANGLE'),
                'MEAN_SOLAR_ZENITH_ANGLE': s2_img.get('MEAN_SOLAR_ZENITH_ANGLE')
            })

        return linked

    # Date filter
    date_min = YEAR_MIN + '-01-01'
    START = ee.Date(date_min)
    date_max = YEAR_MAX + '-01-01'
    END = ee.Date(date_max).advance(1,'year')

    # Spatial and temporal filter with simplified ROI to improve performance
    colFilter = ee.Filter.And(
        ee.Filter.bounds(ROI.simplify(100)),
        ee.Filter.date(START, END))

    # Load collections and apply filters
    S2_SR = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filter(colFilter).select(['B3','B8'])
    DW = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filter(colFilter).select(['water', 'label'])

    # Check if the year is before 2019 to determine which Sentinel-2 collection to use
    if int(YEAR_MIN) < 2019:

        # load Sentinel-2 TOA collection for years before 2019
        S2_TOA = ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filter(colFilter).select(['B3','B8'])

        if int(YEAR_MIN) < 2017:
            col_filtered = DW.linkCollection(S2_TOA, ['B3','B8'], ['MEAN_SOLAR_AZIMUTH_ANGLE', 'MEAN_SOLAR_ZENITH_ANGLE'])
        else:
            # Apply the set_has_data function to each image in the Dynamic World collection
            # This will add a 'has_data' property to each image indicating whether it has data
            DW = DW.map(set_has_data)

            # Filter the Dynamic World collection to only include images with data
            DW = DW.filter(ee.Filter.eq('has_data', True))

            # Link the Dynamic World collection with Sentinel-2 images
            col_filtered = DW.map(get_s2_image)

    else:
        # Link collections
        col_filtered = DW.linkCollection(S2_SR, ['B3','B8'], ['MEAN_SOLAR_AZIMUTH_ANGLE', 'MEAN_SOLAR_ZENITH_ANGLE'])



    # apply water processing mask
    col_water = col_filtered.map(calc_water)

    # apply cloud filtering with CloudScore+ collection
    if FILTER_CLOUDS:
        col_water = apply_cloud_filtering(col_water, ROI_GEO=ROI, DATE_START=START, DATE_END=END, CLOUD_THRESHOLD=CLOUD_THRESHOLD)\
            .select(['water', 'label', 'NDWI', 'water_mask', 'water_combi'])
    else:
        col_water = col_water.select(['water', 'label', 'NDWI', 'water_mask', 'water_combi'])

    return col_water

### Main functions ### ----------------------------------------------------------------------------

def calculate_water_occurrence(
    YEAR,
    AOI,
    FILTER_CLOUDS = True,
    CLOUD_THRESHOLD = 0.60,
    MASK_SHADOWS = False,
    SHADOWS_WO_THRESHOLD = 40,
    INCLUDE_LABEL_MODE = False,
    LABEL_NODATA_VALUE = 255,
):
    """
    Computes monthly and yearly water occurrence statistics for a given year and area of interest (AOI) using Sentinel-2 data.
    This function processes a Sentinel-2-derived water mask collection to calculate:
      - for each month the average water occurrence percentage (as a percentage of valid observations). --> WO_1, WO_2, ..., WO_12
      - for the entire year the average water occurrence percentage ('WO_year')
      - for the entire year the number of water observations ('total_water_count')
      - for the entire year the number of valid observations ('total_observation_count')
    It supports optional cloud and shadow masking to improve result quality.
    Args:
        YEAR (int or str): The year for which to compute water occurrence statistics.
        AOI: Area of interest, either:
            - `ee.Image` mask (non-zero pixels are treated as inside AOI), or
            - `ee.Geometry` / `ee.Feature` / `ee.FeatureCollection`.
        FILTER_CLOUDS (bool, optional): Whether to filter out cloudy observations. Defaults to True.
        CLOUD_THRESHOLD (float, optional): Maximum allowed cloud score+ cs probability for an observation to be considered valid. Defaults to 0.60.
        MASK_SHADOWS (bool, optional): Whether to mask shadowed areas using hillshadow analysis. Defaults to True.
        SHADOWS_WO_THRESHOLD (float, optional): Threshold for water occurrence in shadowed areas to be considered valid. Defaults to 0.
        INCLUDE_LABEL_MODE (bool, optional): When True, append a band named 'DW_label_mode' containing the yearly mode of the Dynamic World label for the ROI.
        LABEL_NODATA_VALUE (int, optional): Value used to fill pixels with no Dynamic World observations when INCLUDE_LABEL_MODE is True. Defaults to 255.
    Returns:
        ee.Image: An Earth Engine image containing:
            - 12 bands for monthly water occurrence percentages (WO_1, WO_2, ..., WO_12).
            - A band for yearly average water occurrence percentage ('WO_year').
            - A band for the number of water observations for the whole year ('total_water_count').
            - A band for the number of valid observations for the whole year ('total_observation_count').
        All bands are clipped to the AOI.
    """

    # Normalize AOI to a geometry (for collection filtering) and a mask image
    # (for updateMask operations).
    aoi_mask, aoi_geom = normalize_aoi(AOI)

    # load collection and apply water processing mask
    col_water = load_s2_dw_water(aoi_geom, YEAR, YEAR, FILTER_CLOUDS=FILTER_CLOUDS, CLOUD_THRESHOLD=CLOUD_THRESHOLD)

    # annual DW mode
    label_mode_image = None
    if INCLUDE_LABEL_MODE:
        label_collection = col_water.select('label')
        label_mode_image = ee.Image(ee.Algorithms.If(
            label_collection.size().gt(0),
            label_collection.mode().toByte(),
            ee.Image.constant(LABEL_NODATA_VALUE).toByte()
        )).rename('DW_LABEL')

        label_mode_image = label_mode_image.updateMask(aoi_mask).unmask(LABEL_NODATA_VALUE, False)


    # Define a function for monthly processing that remains entirely on the server side.
    def process_month(MONTH):
        """Computes monthly water occurrence frequency based on Water Index without further filtering."""
        monthly_col = filter_by_year_and_month(col_water, ee.Number.parse(YEAR), MONTH)

        def process_non_empty_monthly_col(monthly_col):
            """Processes a non-empty monthly collection to compute water occurrence statistics."""

            # count total observations within month (cast to Float for numeric consistency)
            monthly_obs_count = monthly_col.select('water').reduce(ee.Reducer.count()).rename('monthly_observation_count').toFloat()

            # count water observations within month (cast to Float)
            monthly_water_count = monthly_col.select('water_mask').reduce(ee.Reducer.sum()).rename('monthly_water_count').toFloat()

            # create mask with pixels equal to one where at least one water observation is available (Byte)
            monthly_water_mask = monthly_water_count.gt(0).rename('monthly_water_mask').unmask(0).toByte()

            # create mask with pixels equal to one where at least one observation is available (Byte)
            observation_mask = monthly_obs_count.gt(0).rename('monthly_observation_mask').unmask(0).toByte()

            # calculate occurrence of water observations in the month by dividing the water count by the total observations. keep only those cells for which at least one observation exists
            monthly_water_occurrence = monthly_water_count.divide(monthly_obs_count).multiply(100) \
                .rename('water_occurrence_month').toFloat()

            # output image with water occurrence count, total observation count, and water mask
            output = monthly_water_occurrence \
                .addBands(monthly_obs_count) \
                .addBands(monthly_water_count) \
                .addBands(monthly_water_mask) \
                .addBands(observation_mask) \
                .set({'year': YEAR, 'month': MONTH, 'empty': 0,
                      'azimuth': monthly_col.aggregate_mean('MEAN_SOLAR_AZIMUTH_ANGLE'),
                      'zenith': monthly_col.aggregate_mean('MEAN_SOLAR_ZENITH_ANGLE')
                      })

            output = output.updateMask(observation_mask)

            return output

        def create_empty_monthly_image():
            """Creates a placeholder image for an empty month with correct band names and types."""
            # Build an explicit empty placeholder with correct band names and types:
            wo_img = ee.Image.constant(0).toFloat().rename('water_occurrence_month')
            obs_count_img = ee.Image.constant(0).toFloat().rename('monthly_observation_count')
            water_count_img = ee.Image.constant(0).toFloat().rename('monthly_water_count')
            water_mask_img = ee.Image.constant(0).toByte().rename('monthly_water_mask')
            obs_mask_img = ee.Image.constant(0).toByte().rename('monthly_observation_mask')

            empty_img = ee.Image.cat([wo_img, obs_count_img, water_count_img, water_mask_img, obs_mask_img])
            empty_img = empty_img.set({'year': YEAR, 'month': MONTH, 'empty': 1, 'azimuth': 0, 'zenith': 0})
            # Do NOT fully mask (avoid MaskOnly); zeros with correct types keep the collection homogeneous.
            return empty_img

        # Check if the monthly collection is not empty and apply processing.
        # If it is empty, return an image with zero values and a flag indicating emptiness.
        # Return the typed placeholder directly (do not selfMask() as that may create MaskOnly bands).
        result = ee.Algorithms.If(
            monthly_col.size().gt(0),
            process_non_empty_monthly_col(monthly_col),
            create_empty_monthly_image()
        )

        return result

    # Map over serverside list of months to create a nested list of monthly images and flatten the nested list into a flat ee.List of images.
    monthly_collection = ee.ImageCollection(ee.List(ee.List.sequence(1, 12).map(lambda m: process_month(m))))

    # Count total water observations across all months
    water_count = monthly_collection.select('monthly_water_count').sum().rename('total_water_count')

    # Count months with water presence
    water_month_count = monthly_collection.select('monthly_water_mask').sum().rename('water_month_count')

    # create a mask for the pixels with at least one month with water presence
    total_water_mask = water_month_count.gt(0).rename('total_water_mask')

    # Count total observations across all months
    observation_count = monthly_collection.select('monthly_observation_count').sum().rename('total_observation_count').updateMask(total_water_mask)

    # count months with observations
    observation_month_count = monthly_collection.select('monthly_observation_mask').sum().rename('total_observation_mask').updateMask(total_water_mask)

    # calculate yearly occurrence one first time before removing shadows
    yearly_occurrence_sum = monthly_collection.select('water_occurrence_month').sum().rename('water_occurrence_year_sum').selfMask()
    yearly_occurrence = yearly_occurrence_sum.divide(observation_month_count).rename('water_occurrence_year').updateMask(total_water_mask)

    # mask out shadowed areas if requested
    if MASK_SHADOWS:
        # load DEM with crs of first image
        smoothedDEM = load_fabdem(ROI_GEO=aoi_geom.buffer(1000), smooth=True)

        # Define a function to process shadows for each month
        def process_shadows_month(month_img):

            # model shadow mask only if azimuth and zenith are not null and month_img is not empty
            # Return a single-band image named 'monthly_shadow_mask' of type Byte for all branches.
            return ee.Algorithms.If(
                ee.Algorithms.IsEqual(month_img.get('azimuth'), None),
                ee.Image.constant(0).toByte().rename('monthly_shadow_mask').set({'year': month_img.get('year'), 'month': month_img.get('month'), 'empty': 1}),
                ee.Algorithms.If(
                    ee.Algorithms.IsEqual(month_img.get('empty'), 1),
                    ee.Image.constant(0).toByte().rename('monthly_shadow_mask').set({'year': month_img.get('year'), 'month': month_img.get('month'), 'empty': 1}),
                    ee.Image(model_shadow_mask(
                        DEM=smoothedDEM,
                        AZIMUTH=month_img.get('azimuth'),
                        ZENITH=month_img.get('zenith'),
                        expand=True
                    )).rename('monthly_shadow_mask').unmask(0).toByte().set({'year': month_img.get('year'), 'month': month_img.get('month'), 'empty': 0})
                )
            )

        # Map over monthly collection to process shadows for each month and remove shadow areas form the monthly images
        monthly_shadow_collection = ee.ImageCollection(monthly_collection.map(process_shadows_month))

        # remove shadow areas from monthly collection by going through each image and applying the shadow mask
        def apply_shadow_mask(img):
            # Get the shadow mask for the current month
            monthly_shadow_mask = monthly_shadow_collection.filter(ee.Filter.eq('month', img.get('month'))).first()

            # The monthly_shadow_mask may be None or may not contain the expected band
            # (e.g. when the month was empty or azimuth/zenith were missing). Build a
            # safe shadow band of type Byte: if missing, use a zero Byte mask so the month
            # is effectively not shadow-masked rather than throwing an error.
            safe_shadow = ee.Algorithms.If(
                ee.Algorithms.IsEqual(monthly_shadow_mask, None),
                ee.Image.constant(0).toByte().rename('monthly_shadow_mask'),
                ee.Algorithms.If(
                    ee.List(ee.Image(monthly_shadow_mask).bandNames()).contains('monthly_shadow_mask'),
                    ee.Image(monthly_shadow_mask).select('monthly_shadow_mask').unmask(0).toByte(),
                    ee.Image.constant(0).toByte().rename('monthly_shadow_mask')
                )
            )

            safe_shadow = ee.Image(safe_shadow)

            # remove pixels with yearly water occurrence above the threshold from shadow mask
            shadow_mask_wo_removed = safe_shadow.updateMask(
                yearly_occurrence.select('water_occurrence_year').lt(SHADOWS_WO_THRESHOLD)
            ).unmask(0)

            return img.updateMask(shadow_mask_wo_removed.eq(0))

        # Apply the shadow mask to each image in the monthly collection
        monthly_collection = monthly_collection.map(apply_shadow_mask)

        # After applying the shadow mask, we can now safely calculate the yearly occurrence and other statistics.
        # This ensures that shadowed areas do not contribute to the water occurrence statistics.

        # count months with water presence
        water_month_count = monthly_collection.select('monthly_water_mask').sum().rename('water_month_count')

        # create a mask of pixels with at least one month with water presence
        total_water_mask = water_month_count.gt(0).rename('total_water_mask')

        # Count total water observations across all months
        water_count = monthly_collection.select('monthly_water_count').sum().rename('OBS_WATER').updateMask(total_water_mask)

        # Count total observations across all months
        observation_count = monthly_collection.select('monthly_observation_count').sum().rename('OBS_TOTAL').updateMask(total_water_mask)

        # count months with observations
        observation_month_count = monthly_collection.select('monthly_observation_mask').sum().rename('OBS_MONTH').updateMask(total_water_mask)

        # calculate final yearly occurrence
        yearly_occurrence_sum = monthly_collection.select('water_occurrence_month').sum().rename('water_occurrence_year_sum').selfMask()
        yearly_occurrence = yearly_occurrence_sum.divide(observation_month_count).rename('WOF_YEAR').updateMask(total_water_mask)

    # Stack 12 monthly occurrences into a single image to consolidate all monthly water occurrence bands.
    def get_monthly_band(m):
        img = monthly_collection.filter(ee.Filter.eq('month', m)).first()

        # Ensure the output is an image with the correct band name and mask.
        # If img is None, return a zero mask image with the correct band name.
        return ee.Algorithms.If(
            ee.Number(img.get('empty')).eq(1),
            ee.Image().rename(['WOF_' + str(m)]),
            img.select('water_occurrence_month').rename(['WOF_' + str(m)]) #.updateMask(img.select('observations_month_count').gt(0))
        )

    # Create a list of monthly bands for each month from 1 to 12.
    # Each band is named 'WO_1', 'WO_2', ..., 'WO_12' and contains the monthly water occurrence percentage.
    monthly_bands = ee.Image.cat([
        get_monthly_band(m) for m in range(1, 13)
    ])

    # Combine all monthly occurrence bands with the yearly occurrence, the total water observation count and the total observation count.
    base_result = monthly_bands \
        .addBands(yearly_occurrence) \
        .addBands(water_count) \
        .addBands(observation_count)



    result = base_result.updateMask(aoi_mask)

    if INCLUDE_LABEL_MODE and label_mode_image is not None:
        result = ee.Image.cat([result, label_mode_image])

    return result

### Export function ### ----------------------------------------------------------------------------
def get_WO_tiles_and_export(
    ROI,
    YEAR,
    FOLDER_NAME='WO_mapping',
    BASE_FILENAME='WOF',
    EXPORT_CRS='EPSG:4326',
    FILE_SCALE=10,
    FILTER_CLOUDS=True,
    CLOUD_THRESHOLD=0.60,
    MASK_SHADOWS=False,
    SHADOWS_WO_THRESHOLD=0,
    INCLUDE_LABEL_MODE=True
):

    aoi_mask, aoi_geom = normalize_aoi(ROI)

    col_water = calculate_water_occurrence(
                YEAR,
                AOI=aoi_mask,
                FILTER_CLOUDS=FILTER_CLOUDS,
                CLOUD_THRESHOLD=CLOUD_THRESHOLD,
                MASK_SHADOWS=MASK_SHADOWS,
                SHADOWS_WO_THRESHOLD=SHADOWS_WO_THRESHOLD,
                INCLUDE_LABEL_MODE=INCLUDE_LABEL_MODE,
                LABEL_NODATA_VALUE=255,
            )

    image_export = col_water.toByte().unmask(255, False)
    task_name = f"{BASE_FILENAME}_{YEAR}"

    task = ee.batch.Export.image.toDrive(
                image=image_export,
                description='Export_' + task_name,
                folder=str(FOLDER_NAME).strip(),
                fileNamePrefix=task_name,
                region=aoi_geom,
                crs=EXPORT_CRS,
                scale=FILE_SCALE,
                maxPixels=1e13,
                skipEmptyTiles=True,
                formatOptions={'cloudOptimized': True, 'noData': 255},
            )

    task.start()

    print(f"Task started: {task_name} - check the Earth Engine Task Manager for progress.")


## Step 2: Linkage to Google Earth Engine project
A Google Earth Engine account with a corresponding project are required in order to run the workflow

In [4]:
# authenticate and initialize GEE by setting the project name
ee.Authenticate()
ee.Initialize(project = "ee-dw-water")

## Step 3: Definition of study area and year
Provide the study area as GEE asset (either as Feature/FeatureCollection or as Image with pixels of 1 for area to be analyzed and 0 as no-data value), and define the year, the coordinate reference system, and output name.

In [5]:
# load Area of Interest
AOI = ee.Image("projects/ee-dw-water/assets/YOURASSET")
YEAR = '2025'
BASE_FILENAME='WOF_YOURASSET'
CRS='EPSG:4326'

## 3 - Visualize - *only for relatively small regions*

In [ ]:
#  create geemap object
m = geemap.Map()
m.centerObject(AOI, 12)

# load water occurrence layer
col_water = calculate_water_occurrence(
    AOI=AOI,
    YEAR=YEAR,
    FILTER_CLOUDS= True,
    MASK_SHADOWS=True,
    SHADOWS_WO_THRESHOLD=40
    )

# visualization parameters
water_prob_vis = {
  'min': 1, 'max': 100,
  'palette': ['#bddf26', '#7ad151', '#44bf70', '#22a884', '#21918c', '#2a788e', '#355f8d', '#414487', '#482475', '#440154']
}

# add AOI layer
m.add_layer(AOI, {}, 'AOI')

# add layers
m.add_layer(col_water.select('WOF_YEAR'), water_prob_vis, 'yearly occurrence') # Yearly occurrence (WOF_YEAR)
# m.add_layer(col_water.select('WO_5'), water_prob_vis, '5 occurrence') # monthly Occurrence (here WO_5 for May)


# show map
m

## 4 - Export to Gooogle Drive

In [ ]:
# run the export function
get_WO_tiles_and_export(
    AOI,
    YEAR=YEAR,
    BASE_FILENAME=BASE_FILENAME,
    FOLDER_NAME=f"{BASE_FILENAME}_{YEAR}",
    FILTER_CLOUDS = True,
    MASK_SHADOWS = True,
    INCLUDE_LABEL_MODE=True,
    EXPORT_CRS=CRS
    )